# 2.6 – Classification Dataset Preparation

Converts the regression datasets into binary classification datasets by deriving a **`NO_Benefit`** label from `health_gain`.

**MCID (Minimum Clinically Important Difference) = 7** for Oxford Knee Score.

| `health_gain` | `NO_Benefit` | Interpretation |
|---|---|---|
| ≥ 7 | 0 | Patient *did* benefit |
| < 7 | 1 | Patient did **not** benefit |

**Source datasets:**
- `prepared_datasets/pipeline1_train.parquet` / `pipeline1_test.parquet`
- `data/interim/2.1-train.parquet` / `2.1-test.parquet`

**Output:**
- `prepared_datasets/2.6-pipeline1-train.parquet`
- `prepared_datasets/2.6-pipeline1-test.parquet`
- `data/interim/2.6-train.parquet`
- `data/interim/2.6-test.parquet`

In [1]:
import polars as pl
from pathlib import Path

## 1. Load source datasets

In [2]:
# Pipeline-1 datasets (imputed, encoded, ready for modelling)
pipeline1_train = pl.read_parquet('./prepared_datasets/pipeline1_train.parquet')
pipeline1_test  = pl.read_parquet('./prepared_datasets/pipeline1_test.parquet')

# 2.1 datasets (feature-engineered, before imputation/encoding)
train_21 = pl.read_parquet('./data/interim/2.1-train.parquet')
test_21  = pl.read_parquet('./data/interim/2.1-test.parquet')

print('pipeline1_train :', pipeline1_train.shape)
print('pipeline1_test  :', pipeline1_test.shape)
print('2.1-train       :', train_21.shape)
print('2.1-test        :', test_21.shape)

pipeline1_train : (86955, 43)
pipeline1_test  : (21739, 43)
2.1-train       : (74400, 48)
2.1-test        : (36312, 48)


## 2. Inspect `health_gain` distribution

In [3]:
MCID = 7

for name, df in [
    ('pipeline1_train', pipeline1_train),
    ('pipeline1_test',  pipeline1_test),
    ('2.1-train',       train_21),
    ('2.1-test',        test_21),
]:
    hg = df['health_gain'].drop_nulls()
    benefit     = (hg >= MCID).sum()
    no_benefit  = (hg <  MCID).sum()
    total       = len(hg)
    print(f"{name:<20}  benefit (≥{MCID}): {benefit:>6} ({benefit/total*100:.1f}%)  "
          f"no-benefit (<{MCID}): {no_benefit:>6} ({no_benefit/total*100:.1f}%)")

pipeline1_train       benefit (≥7):  74175 (85.3%)  no-benefit (<7):  12780 (14.7%)
pipeline1_test        benefit (≥7):  18689 (86.0%)  no-benefit (<7):   3050 (14.0%)
2.1-train             benefit (≥7):  63481 (85.3%)  no-benefit (<7):  10919 (14.7%)
2.1-test              benefit (≥7):  31243 (86.0%)  no-benefit (<7):   5069 (14.0%)


## 3. Add `NO_Benefit` label and drop `health_gain`

- **0** → patient benefited (`health_gain ≥ MCID`)
- **1** → patient did NOT benefit (`health_gain < MCID`)

In [4]:
def add_classification_label(df: pl.DataFrame, mcid: int = 7) -> pl.DataFrame:
    """Add NO_Benefit binary label and drop health_gain."""
    return (
        df
        .with_columns(
            pl.when(pl.col('health_gain') >= mcid)
              .then(pl.lit(0))
              .otherwise(pl.lit(1))
              .cast(pl.Int8)
              .alias('NO_Benefit')
        )
        .drop('health_gain')
    )


pipeline1_train_clf = add_classification_label(pipeline1_train, MCID)
pipeline1_test_clf  = add_classification_label(pipeline1_test,  MCID)
train_21_clf        = add_classification_label(train_21,        MCID)
test_21_clf         = add_classification_label(test_21,         MCID)

print('Output shapes:')
print('  pipeline1_train_clf :', pipeline1_train_clf.shape)
print('  pipeline1_test_clf  :', pipeline1_test_clf.shape)
print('  2.1-train_clf       :', train_21_clf.shape)
print('  2.1-test_clf        :', test_21_clf.shape)

print()
print('health_gain removed  :', 'health_gain' not in pipeline1_train_clf.columns)
print('NO_Benefit present   :', 'NO_Benefit'  in pipeline1_train_clf.columns)

Output shapes:
  pipeline1_train_clf : (86955, 43)
  pipeline1_test_clf  : (21739, 43)
  2.1-train_clf       : (74400, 48)
  2.1-test_clf        : (36312, 48)

health_gain removed  : True
NO_Benefit present   : True


## 4. Verify class balance

In [5]:
for name, df in [
    ('pipeline1_train (clf)', pipeline1_train_clf),
    ('pipeline1_test  (clf)', pipeline1_test_clf),
    ('2.1-train (clf)',       train_21_clf),
    ('2.1-test  (clf)',       test_21_clf),
]:
    counts = df['NO_Benefit'].value_counts().sort('NO_Benefit')
    total  = df.shape[0]
    rows   = counts.to_dicts()
    label_map = {0: 'Benefit (0)', 1: 'No Benefit (1)'}
    print(f"\n{name}")
    for row in rows:
        lbl = row['NO_Benefit']
        cnt = row['count']
        print(f"  {label_map.get(lbl, lbl):<20}: {cnt:>6}  ({cnt/total*100:.1f}%)")


pipeline1_train (clf)
  Benefit (0)         :  74175  (85.3%)
  No Benefit (1)      :  12780  (14.7%)

pipeline1_test  (clf)
  Benefit (0)         :  18689  (86.0%)
  No Benefit (1)      :   3050  (14.0%)

2.1-train (clf)
  Benefit (0)         :  63481  (85.3%)
  No Benefit (1)      :  10919  (14.7%)

2.1-test  (clf)
  Benefit (0)         :  31243  (86.0%)
  No Benefit (1)      :   5069  (14.0%)


## 5. Save classification datasets

In [6]:
# Output paths
Path('./prepared_datasets').mkdir(parents=True, exist_ok=True)
Path('./data/interim').mkdir(parents=True, exist_ok=True)

pipeline1_train_clf.write_parquet('./prepared_datasets/2.6-pipeline1-train.parquet', compression='gzip')
pipeline1_test_clf.write_parquet( './prepared_datasets/2.6-pipeline1-test.parquet',  compression='gzip')
train_21_clf.write_parquet(       './data/interim/2.6-train.parquet',                compression='gzip')
test_21_clf.write_parquet(        './data/interim/2.6-test.parquet',                 compression='gzip')

print('Saved:')
print('  prepared_datasets/2.6-pipeline1-train.parquet')
print('  prepared_datasets/2.6-pipeline1-test.parquet')
print('  data/interim/2.6-train.parquet')
print('  data/interim/2.6-test.parquet')

Saved:
  prepared_datasets/2.6-pipeline1-train.parquet
  prepared_datasets/2.6-pipeline1-test.parquet
  data/interim/2.6-train.parquet
  data/interim/2.6-test.parquet
